# Notebook 7 — Regresor LSTM de Precios → MongoDB
### Investing AI · iDeSo · UNMSM

**Objetivo:** Leer datos de `precios_ohlcv` en MongoDB, construir secuencias temporales,
entrenar un **regresor LSTM** que predice el **precio de cierre continuo** (en USD) del día
siguiente, proyectar precios futuros a 7/14/30/60 días con bandas de confianza, y guardar
predicciones y métricas en las colecciones `predicciones` y `metricas_modelos`.

**Arquitectura:** `LSTM(64) → Dropout(0.2) → LSTM(32) → Dropout(0.2) → Dense(16, relu) → Dense(1, linear)`
**Entrenamiento:** 100 épocas · batch_size=32 · Adam(lr=0.001) · loss=`mse` · EarlyStopping
**Ventana temporal:** `N_STEPS = 60` días

**Prerequisito:** Haber ejecutado el Notebook 1 (colección `precios_ohlcv` poblada).
**Salida:** Colecciones `predicciones` (`modelo='LSTM_REG'`) y `metricas_modelos` (`modelo='LSTM_REG'`)
listas para la API, además de un archivo `datos_lstm_reg.json` exportado para el frontend.

> A diferencia del Notebook 3 (clasificador BUY/SELL), este notebook resuelve un problema de
> **regresión**: no predice una categoría, sino un valor numérico continuo (el precio en USD).
> Por eso no hay `class_weight` ni `sigmoid`; la capa de salida es `Dense(1, activation='linear')`
> y la función de pérdida es `mse` en vez de `binary_crossentropy`.


## Paso 1 — Instalación de librerías

In [ ]:
# Instalación de librerías necesarias
# TensorFlow ya viene preinstalado en Colab; se asegura scikit-learn, pymongo y statsmodels
# (statsmodels es opcional: solo se usa para el baseline ARIMA de comparación del Paso 9)
!pip install 'pymongo[srv]' scikit-learn statsmodels --quiet
print('✓ Librerías instaladas')

## Paso 2 — Conexión a MongoDB Atlas

In [ ]:
# Conectar a MongoDB Atlas usando el Secret MONGO_URI de Colab
# (Secrets: ícono de llave en el panel izquierdo de Colab)
from google.colab import userdata
from pymongo import MongoClient
import pandas as pd
import numpy as np
from datetime import datetime
import json
import random

MONGO_URI = userdata.get('MONGO_URI')
client    = MongoClient(MONGO_URI)
db        = client['spbi']

# Verificar conexión
client.admin.command('ping')
print('✓ Conectado a MongoDB Atlas')
print(f'  Base de datos: spbi')
print(f'  Colecciones disponibles: {db.list_collection_names()}')

## Paso 3 — Semilla fija y configuración de reproducibilidad

In [ ]:
# SEED fija para reproducibilidad (numpy, python, tensorflow)
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'✓ Semilla fija SEED={SEED} aplicada (random, numpy, tensorflow)')
print(f'  TensorFlow versión: {tf.__version__}')

## Paso 4 — Carga de datos desde MongoDB

In [ ]:
# Función para cargar precios OHLCV de un ticker desde MongoDB
# (idéntica a la del Notebook 3: NO se descarga nada de Internet, solo se lee de la BD)
def cargar_datos(ticker):
    """
    Lee los documentos de precios_ohlcv para un ticker
    y los devuelve como DataFrame ordenado por fecha.
    """
    docs = list(db['precios_ohlcv'].find({'ticker': ticker}).sort('fecha', 1))
    if not docs:
        return pd.DataFrame()
    df = pd.DataFrame(docs)
    df['fecha'] = pd.to_datetime(df['fecha'])
    df = df.sort_values('fecha').reset_index(drop=True)
    return df

# Verificación rápida con BVN
df_test = cargar_datos('BVN')
print(f'✓ BVN: {len(df_test)} registros cargados')
print(df_test[['fecha','close','sma_20','rsi_14']].tail(3).to_string(index=False))

## Paso 5 — Ingeniería de features y secuencias temporales (regresión)

**Nota técnica:** al ser un problema de **regresión** (no de clasificación), el target ya no es
0/1 sino el **precio de cierre real del día siguiente** (`close.shift(-1)`). Se usan dos
`MinMaxScaler` independientes — uno para las features (`scaler_X`) y otro para el target
(`scaler_y`) — ambos ajustados **solamente** con la partición de entrenamiento, para evitar
*data leakage* y para poder des-escalar las predicciones de vuelta a USD.

Para `VOLCABC1.LM` (precio casi constante) el target sigue siendo el precio continuo: a
diferencia del clasificador (Notebook 3), aquí no hace falta un target alternativo basado en
volumen, porque un regresor no depende de que haya una dirección clara de cambio — simplemente
aprende a reproducir la serie de precios, que para este ticker es una serie casi plana (y eso
es precisamente lo que se espera que el modelo aprenda).


In [ ]:
from sklearn.preprocessing import MinMaxScaler

N_STEPS = 60  # tamaño de la ventana temporal (días) que ve el LSTM en cada secuencia

# Features de precio e indicadores técnicos (se usan para los 5 tickers, incluido VOLCABC1.LM)
FEATURES = ['close', 'sma_20', 'sma_50', 'ema_12', 'ema_26', 'rsi_14', 'retorno']

def preparar_features(df):
    """
    Feature engineering para el regresor de precios.
    Target: precio de cierre REAL del día siguiente (valor continuo en USD),
    NO una categoría BUY/SELL como en el Notebook 3.
    """
    df = df.copy()
    df['retorno'] = df['close'].pct_change()
    # Target de regresión: precio de cierre de mañana
    df['target'] = df['close'].shift(-1)
    df = df.dropna(subset=FEATURES + ['target'])

    feats   = df[FEATURES].values
    target  = df['target'].values
    fechas  = df['fecha'].values
    precios = df['close'].values
    return feats, target, fechas, precios

def crear_secuencias(feats, target, fechas, precios, n_steps=N_STEPS):
    """
    Convierte una matriz de features 2D (muestras, features) en
    secuencias 3D (muestras, n_steps, features) mediante ventana deslizante.
    El target de cada secuencia es el precio de cierre de mañana, tomando como
    referencia el último día de la ventana (día i).
    """
    X_seq, y_seq, fechas_seq, precios_seq = [], [], [], []
    for i in range(n_steps, len(feats)):
        X_seq.append(feats[i - n_steps:i])
        y_seq.append(target[i])
        fechas_seq.append(fechas[i])
        precios_seq.append(precios[i])
    return (np.array(X_seq), np.array(y_seq),
            np.array(fechas_seq), np.array(precios_seq))

print(f'✓ Funciones de feature engineering y secuencias definidas (N_STEPS={N_STEPS})')

## Paso 6 — Construcción del modelo LSTM Regressor (Keras Sequential)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

def construir_modelo_lstm_reg(n_steps, n_features):
    """
    Arquitectura LSTM para regresión de precios continuos:
    LSTM(64) -> Dropout(0.2) -> LSTM(32) -> Dropout(0.2) -> Dense(16, relu) -> Dense(1, linear)
    """
    modelo = Sequential([
        Input(shape=(n_steps, n_features)),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1, activation='linear')
    ])

    modelo.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    return modelo

print('✓ Función construir_modelo_lstm_reg definida')
print('  Arquitectura: LSTM(64) -> Dropout(0.2) -> LSTM(32) -> Dropout(0.2) -> Dense(16,relu) -> Dense(1,linear)')

## Paso 7 — Entrenamiento y evaluación del LSTM Regressor

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def entrenar_lstm_reg(X, y):
    """
    Entrena el LSTM Regressor sobre una partición temporal ESTRICTA 80/20 (sin data leakage).
    - scaler_X ajustado SOLO con features de entrenamiento (aplicado feature a feature)
    - scaler_y ajustado SOLO con el target de entrenamiento (para des-escalar a USD luego)
    - 100 épocas, batch_size=32, Adam(lr=0.001), loss='mse'
    - EarlyStopping sobre val_loss para evitar sobreajuste
    """
    n     = len(X)
    corte = int(n * 0.80)

    n_steps, n_features = X.shape[1], X.shape[2]

    # ── Escalado de features: ajustado sobre train aplanado (2D) ──────────
    scaler_X = MinMaxScaler()
    X_train_flat = X[:corte].reshape(-1, n_features)
    scaler_X.fit(X_train_flat)

    def escalar_X(X_):
        forma = X_.shape
        X_flat = X_.reshape(-1, n_features)
        X_esc  = scaler_X.transform(X_flat)
        return X_esc.reshape(forma)

    X_train, X_test = escalar_X(X[:corte]), escalar_X(X[corte:])

    # ── Escalado del target (precio continuo) ──────────────────────────────
    scaler_y = MinMaxScaler()
    scaler_y.fit(y[:corte].reshape(-1, 1))

    y_train_esc = scaler_y.transform(y[:corte].reshape(-1, 1)).flatten()
    y_test_esc  = scaler_y.transform(y[corte:].reshape(-1, 1)).flatten()
    y_test_usd  = y[corte:]  # valores reales en USD, usados para las métricas finales

    modelo = construir_modelo_lstm_reg(n_steps, n_features)

    early_stop = EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=0
    )

    historial = modelo.fit(
        X_train, y_train_esc,
        epochs=100,
        batch_size=32,
        validation_split=0.15,
        callbacks=[early_stop],
        verbose=0
    )

    # ── Predicción sobre test y des-escalado a USD ─────────────────────────
    y_pred_esc = modelo.predict(X_test, verbose=0).flatten()
    y_pred_usd = scaler_y.inverse_transform(y_pred_esc.reshape(-1, 1)).flatten()

    rmse_usd     = float(np.sqrt(mean_squared_error(y_test_usd, y_pred_usd)))
    precio_medio = float(np.mean(y_test_usd))
    rmse_pct     = round(rmse_usd / precio_medio * 100, 4) if precio_medio else None
    mae_usd      = float(mean_absolute_error(y_test_usd, y_pred_usd))
    r2           = float(r2_score(y_test_usd, y_pred_usd))

    metricas = {
        'rmse_usd'          : round(rmse_usd, 4),
        'rmse_pct'          : rmse_pct,
        'mae_usd'           : round(mae_usd, 4),
        'r2'                : round(r2, 4),
        'precio_medio_test' : round(precio_medio, 4),
        'epocas_entrenadas' : int(len(historial.history['loss'])),
        'n_train'           : int(len(X_train)),
        'n_test'            : int(len(X_test))
    }

    hist_epocas = {
        'loss'     : [round(float(v), 6) for v in historial.history['loss']],
        'mae'      : [round(float(v), 6) for v in historial.history['mae']],
        'val_loss' : [round(float(v), 6) for v in historial.history.get('val_loss', [])],
        'val_mae'  : [round(float(v), 6) for v in historial.history.get('val_mae', [])]
    }

    return modelo, scaler_X, scaler_y, metricas, hist_epocas, y_test_usd, y_pred_usd

print('✓ Función entrenar_lstm_reg definida')
print('  - scaler_X y scaler_y ajustados solo con train (sin leakage)')
print('  - EarlyStopping sobre val_loss, patience=10')
print('  - Métricas en USD des-escaladas con scaler_y.inverse_transform')

## Paso 8 — Predicción a futuro (7 / 14 / 30 / 60 días) con bandas de confianza

Se implementa una predicción **recursiva/iterativa**: en cada paso se predice el precio del
día siguiente y la ventana se desplaza un día, insertando el precio predicho en la posición
de `close` y conservando los demás indicadores técnicos del último día conocido. Esta es una
aproximación razonable a corto/mediano plazo — recalcular SMA/EMA/RSI exactos requeriría
simular el histórico completo día por día, lo que excede el alcance de este notebook.

Las bandas de confianza se calculan como `±1.96 * RMSE`, ensanchándose gradualmente con el
horizonte (`√(h / horizonte_máximo)`) para reflejar la mayor incertidumbre de una predicción
recursiva a más días.


In [ ]:
def predecir_futuro(modelo, scaler_X, scaler_y, ultima_ventana_raw, rmse_usd,
                     horizontes=(7, 14, 30, 60)):
    """
    Predicción recursiva/iterativa a futuro.
    ultima_ventana_raw: array (n_steps, n_features) SIN escalar, con los últimos N_STEPS días.
    Devuelve:
      - serie_diaria: lista con el precio proyectado día a día hasta el horizonte máximo
      - bandas: dict con precio estimado y banda de confianza (±1.96*RMSE) para cada horizonte
    """
    n_features = ultima_ventana_raw.shape[1]
    max_h = max(horizontes)

    # Escalar la ventana inicial con el scaler de features ya ajustado
    ventana_esc = scaler_X.transform(ultima_ventana_raw).reshape(1, ultima_ventana_raw.shape[0], n_features)

    serie_diaria = []
    for _ in range(max_h):
        pred_esc = float(modelo.predict(ventana_esc, verbose=0)[0][0])
        pred_usd = float(scaler_y.inverse_transform([[pred_esc]])[0][0])
        serie_diaria.append(round(pred_usd, 4))

        # Desplazar ventana: se descarta el día más antiguo y se agrega el nuevo día
        # predicho. Los indicadores técnicos (sma/ema/rsi/retorno) se mantienen
        # iguales al último día conocido, ya que no se recalculan recursivamente.
        nueva_fila = ventana_esc[0, -1, :].copy()
        nueva_fila[0] = pred_esc  # posición 0 = 'close' (ver lista FEATURES)
        ventana_esc = np.concatenate(
            [ventana_esc[:, 1:, :], nueva_fila.reshape(1, 1, n_features)], axis=1
        )

    bandas = {}
    for h in horizontes:
        precio_h = serie_diaria[h - 1]
        banda    = 1.96 * rmse_usd * np.sqrt(h / max_h)
        bandas[f'{h}d'] = {
            'precio_estimado'  : precio_h,
            'banda_inferior'   : round(precio_h - banda, 4),
            'banda_superior'   : round(precio_h + banda, 4)
        }

    return serie_diaria, bandas

print('✓ Función predecir_futuro definida (horizontes 7/14/30/60 días, bandas ±1.96*RMSE)')

## Paso 9 — Baseline ARIMA (opcional, para comparación)

Se entrena un modelo `ARIMA(5,1,0)` simple sobre la misma partición temporal, únicamente para
comparar su RMSE contra el del LSTM. Si `statsmodels` no está disponible o el ajuste falla
(por ejemplo, por una serie casi constante como `VOLCABC1.LM`), se omite sin detener el
pipeline.


In [ ]:
def comparar_arima(precios_train, precios_test):
    """
    Baseline simple ARIMA(5,1,0) para contrastar el RMSE del LSTM.
    Devuelve el RMSE en USD del ARIMA, o None si no se pudo entrenar.
    """
    try:
        from statsmodels.tsa.arima.model import ARIMA
        modelo_arima = ARIMA(precios_train, order=(5, 1, 0)).fit()
        pred = modelo_arima.forecast(steps=len(precios_test))
        rmse_arima = float(np.sqrt(mean_squared_error(precios_test, pred)))
        return round(rmse_arima, 4)
    except Exception as e:
        print(f'    (ARIMA baseline no disponible para este ticker: {e})')
        return None

print('✓ Función comparar_arima definida (baseline opcional)')

## Paso 10 — Pipeline completo por ticker

In [ ]:
def procesar_ticker(ticker):
    """
    Pipeline completo para un ticker:
    1. Carga datos de MongoDB
    2. Prepara features + target continuo (precio de mañana)
    3. Construye secuencias temporales de tamaño N_STEPS
    4. Entrena el LSTM Regressor sobre partición temporal 80/20
    5. Proyecta precios futuros (7/14/30/60 días) con bandas de confianza
    6. Compara opcionalmente contra un baseline ARIMA
    7. Guarda predicción + proyecciones + métricas en MongoDB
    """
    print(f'\n  [{ticker}] Procesando...')

    df = cargar_datos(ticker)
    if len(df) < 150:
        print(f'  [{ticker}] ✗ Datos insuficientes ({len(df)} registros) — se omite')
        return None

    feats, target, fechas, precios = preparar_features(df)
    X, y, fechas_seq, precios_seq = crear_secuencias(feats, target, fechas, precios, N_STEPS)

    if len(X) < 50:
        print(f'  [{ticker}] ✗ Secuencias insuficientes tras limpieza — se omite')
        return None

    print(f'  [{ticker}] Secuencias: {len(X)} | N_STEPS={N_STEPS} | features={len(FEATURES)}')

    modelo, scaler_X, scaler_y, metricas, hist_epocas, y_test_usd, y_pred_usd = entrenar_lstm_reg(X, y)

    # ── Baseline ARIMA opcional (sobre la misma partición 80/20) ───────────
    corte = int(len(precios_seq) * 0.80)
    rmse_arima = comparar_arima(precios_seq[:corte], precios_seq[corte:])

    # ── Proyección a futuro usando la última ventana disponible ────────────
    ultima_ventana_raw = feats[-N_STEPS:]
    serie_diaria, bandas_futuro = predecir_futuro(
        modelo, scaler_X, scaler_y, ultima_ventana_raw,
        rmse_usd=metricas['rmse_usd'], horizontes=(7, 14, 30, 60)
    )

    # ── Historial de predicciones vs. valores reales (partición de test) ───
    historico_predicciones = [
        {
            'fecha'         : str(fechas_seq[corte + i])[:10],
            'precio_real'   : round(float(y_test_usd[i]), 4),
            'precio_predicho': round(float(y_pred_usd[i]), 4)
        }
        for i in range(len(y_test_usd))
    ]

    # ── Guardar predicción actual + proyecciones en MongoDB ────────────────
    ultimo_precio = float(precios_seq[-1])
    db['predicciones'].delete_many({'ticker': ticker, 'modelo': 'LSTM_REG'})
    db['predicciones'].insert_one({
        'ticker'                 : ticker,
        'modelo'                 : 'LSTM_REG',
        'ultimo_precio'          : round(ultimo_precio, 4),
        'proyeccion_1d'          : serie_diaria[0],
        'proyecciones_horizonte' : bandas_futuro,          # 7d / 14d / 30d / 60d
        'serie_diaria_60d'       : serie_diaria,            # proyección día a día
        'historico_predicciones' : historico_predicciones[-60:],
        'fecha_prediccion'       : datetime.now().strftime('%Y-%m-%d'),
        'n_steps'                : N_STEPS,
        'created_at'             : datetime.now()
    })

    # ── Guardar métricas en MongoDB ──────────────────────────────────────────
    db['metricas_modelos'].delete_many({'ticker': ticker, 'modelo': 'LSTM_REG'})
    db['metricas_modelos'].insert_one({
        'ticker'             : ticker,
        'modelo'             : 'LSTM_REG',
        **metricas,
        'rmse_arima_baseline': rmse_arima,
        'historial_epocas'   : hist_epocas,
        'n_steps'            : N_STEPS,
        'fecha_entrenamiento': datetime.now()
    })

    rmse_u = metricas['rmse_usd']
    rmse_p = metricas['rmse_pct']
    r2v    = metricas['r2']
    ep     = metricas['epocas_entrenadas']
    comp   = f'| ARIMA_rmse={rmse_arima}' if rmse_arima is not None else ''
    print(f'  [{ticker}] ✓ RMSE=${rmse_u} ({rmse_p}%) | MAE=${metricas["mae_usd"]} | R2={r2v} | épocas={ep} {comp}')

    return {
        'ticker': ticker,
        'ultimo_precio': round(ultimo_precio, 4),
        'metricas': metricas,
        'rmse_arima_baseline': rmse_arima,
        'proyecciones_horizonte': bandas_futuro,
        'serie_diaria_60d': serie_diaria,
        'historial_epocas': hist_epocas,
        'historico_predicciones': historico_predicciones[-60:]
    }

print('✓ Función procesar_ticker definida')

## Paso 11 — Ejecutar pipeline para los 5 tickers

In [ ]:
TICKERS = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']

print('=' * 55)
print('  REGRESOR LSTM DE PRECIOS — ENTRENAMIENTO PARA 5 TICKERS')
print('=' * 55)

resultados = {}
for ticker in TICKERS:
    try:
        res = procesar_ticker(ticker)
        if res is not None:
            resultados[ticker] = res
    except Exception as e:
        print(f'  [{ticker}] ✗ Error inesperado: {e}')

print()
print('=' * 55)
print(f'  ✅ Pipeline completo: {len(resultados)}/{len(TICKERS)} tickers procesados')
print('=' * 55)

## Paso 12 — Verificación en MongoDB

In [ ]:
print('PREDICCIONES LSTM_REG en MongoDB')
print('-' * 55)
for doc in db['predicciones'].find({'modelo': 'LSTM_REG'}, {'_id':0,'historico_predicciones':0,'serie_diaria_60d':0,'created_at':0}):
    t   = doc['ticker']
    up  = doc['ultimo_precio']
    p1  = doc['proyeccion_1d']
    p30 = doc['proyecciones_horizonte']['30d']['precio_estimado']
    print(f'  {t:<15} último=${up:<10} | +1d=${p1:<10} | +30d=${p30}')

print()
print('MÉTRICAS LSTM_REG en MongoDB')
print('-' * 55)
for doc in db['metricas_modelos'].find({'modelo': 'LSTM_REG'}, {'_id':0,'historial_epocas':0,'fecha_entrenamiento':0}):
    t    = doc['ticker']
    ru   = doc['rmse_usd']
    rp   = doc['rmse_pct']
    mae  = doc['mae_usd']
    r2   = doc['r2']
    arim = doc.get('rmse_arima_baseline')
    print(f'  {t:<15} RMSE=${ru:<8} ({rp}%) | MAE=${mae:<8} | R2={r2:<7} | ARIMA_RMSE={arim}')

print()
print('✅ Colecciones predicciones y metricas_modelos (LSTM_REG) verificadas')

## Paso 13 — Exportación a JSON para el frontend (`datos_lstm_reg.json`)

In [ ]:
# Estructura de exportación pensada para el módulo de regresión de precios del frontend
export_data = {
    'modelo'       : 'LSTM_REG',
    'arquitectura' : 'LSTM(64) -> Dropout(0.2) -> LSTM(32) -> Dropout(0.2) -> Dense(16,relu) -> Dense(1,linear)',
    'n_steps'      : N_STEPS,
    'seed'         : SEED,
    'fecha_export' : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'tickers'      : {}
}

for ticker, res in resultados.items():
    export_data['tickers'][ticker] = {
        'ultimo_precio'          : res['ultimo_precio'],
        'metricas'               : res['metricas'],
        'rmse_arima_baseline'    : res['rmse_arima_baseline'],
        'proyecciones_horizonte' : res['proyecciones_horizonte'],
        'serie_diaria_60d'       : res['serie_diaria_60d'],
        'historial_epocas'       : res['historial_epocas'],
        'historico_predicciones' : res['historico_predicciones']
    }

with open('datos_lstm_reg.json', 'w', encoding='utf-8') as f:
    json.dump(export_data, f, ensure_ascii=False, indent=2)

print(f'✓ Archivo datos_lstm_reg.json exportado ({len(resultados)} tickers)')
print(f'  Tamaño aproximado: {len(json.dumps(export_data)) / 1024:.1f} KB')

# Descargar el archivo desde Colab (opcional)
try:
    from google.colab import files
    files.download('datos_lstm_reg.json')
    print('✓ Descarga iniciada en el navegador')
except Exception as e:
    print(f'  (Descarga automática no disponible en este entorno: {e})')

## Resultado

Las colecciones `predicciones` y `metricas_modelos` contienen resultados reales del regresor
LSTM (`modelo='LSTM_REG'`):
- **5 tickers** procesados con la misma arquitectura y ventana de `N_STEPS=60` días.
- **Target continuo** (precio en USD del día siguiente), no una categoría BUY/SELL.
- **Dos scalers independientes** (`scaler_X`, `scaler_y`) ajustados solo con train → sin data leakage.
- **Proyección recursiva** a 7/14/30/60 días con bandas de confianza `±1.96*RMSE`.
- **Métricas**: RMSE (USD y %), MAE y R² sobre la partición de test.
- **Baseline ARIMA(5,1,0)** opcional para contextualizar el desempeño del LSTM.
- **SEED=42** fija → resultados reproducibles.
- **datos_lstm_reg.json** exportado para consumo directo del módulo de regresión del frontend.

**La API (Notebook 4)** puede leer estas colecciones (filtrando `modelo='LSTM_REG'`) y servirlas
al frontend vía HTTP, en paralelo a las predicciones del SVC (`modelo='SVC'`) y del clasificador
LSTM (`modelo='LSTM'`).

**Pipeline completo:** `precios_ohlcv` → LSTM Regressor → `predicciones` + `metricas_modelos` +
`datos_lstm_reg.json` ✓
